# Pulso — Fase 2: Extração de Sintomas e Sugestão de Diagnóstico

**Projeto:** Pulso — Inteligência Cardiológica com Consciência de Dados  
**Autor:** Richard Schmitz | RM 567951  
**Disciplina:** Inteligência Artificial — FIAP | Turma 2TIAOR-2026  

---

## Objetivo

Este notebook implementa a Parte 1 da Fase 2 do projeto Pulso.  
A partir de relatos textuais de pacientes, o sistema:

1. Lê as frases do arquivo `sintomas_pacientes.txt`
2. Identifica sintomas com base no mapa de conhecimento (`mapa_conhecimento.csv`)
3. Sugere um possível diagnóstico para cada relato
4. Analisa se há diferença na identificação entre sintomas típicos (masculinos) e atípicos (femininos)

O eixo de viés de gênero documentado na Fase 1 orienta a construção dos dados e a análise dos resultados.

## 1. Importações e configuração

In [ ]:
import pandas as pd
import re
from pathlib import Path

# Caminhos dos arquivos
# Se estiver no Google Colab, ajuste os caminhos após fazer upload dos arquivos
CAMINHO_FRASES = '../data/sintomas_pacientes.txt'
CAMINHO_MAPA   = '../data/mapa_conhecimento.csv'

print('Bibliotecas carregadas com sucesso.')

## 2. Leitura dos relatos de pacientes

In [ ]:
def carregar_frases(caminho):
    """Lê o arquivo de relatos e retorna apenas as linhas numeradas."""
    frases = []
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            # Ignora comentários e linhas vazias
            if linha and not linha.startswith('#'):
                # Remove o número inicial (ex: "1. ")
                texto = re.sub(r'^\d+\.\s*', '', linha)
                frases.append(texto)
    return frases

frases = carregar_frases(CAMINHO_FRASES)

print(f'Total de relatos carregados: {len(frases)}\n')
for i, frase in enumerate(frases, 1):
    print(f'Paciente {i:02d}: {frase}')

## 3. Carregamento do mapa de conhecimento

In [ ]:
mapa = pd.read_csv(CAMINHO_MAPA)

print(f'Mapa de conhecimento: {len(mapa)} associações carregadas\n')
mapa.head(10)

## 4. Motor de extração de sintomas e sugestão de diagnóstico

A lógica funciona em três etapas:

1. Para cada relato, verifica quais sintomas do mapa estão presentes no texto
2. Conta quantos sintomas de cada doença foram encontrados
3. Sugere a doença com maior número de correspondências

In [ ]:
def normalizar(texto):
    """Converte para minúsculas e remove pontuação para comparação."""
    return texto.lower().strip()


def extrair_sintomas(frase, mapa):
    """
    Verifica quais sintomas do mapa estão presentes na frase.
    Retorna lista de sintomas encontrados.
    """
    frase_norm = normalizar(frase)
    sintomas_encontrados = []

    colunas_sintoma = ['sintoma_1', 'sintoma_2', 'sintoma_3']
    for _, linha in mapa.iterrows():
        for col in colunas_sintoma:
            sintoma = normalizar(str(linha[col]))
            if sintoma in frase_norm:
                sintomas_encontrados.append(sintoma)

    return list(set(sintomas_encontrados))


def sugerir_diagnostico(frase, mapa):
    """
    Conta correspondências por doença e retorna a mais provável.
    Retorna diagnóstico, nível de risco, perfil predominante e sintomas encontrados.
    """
    frase_norm = normalizar(frase)
    pontuacao = {}
    colunas_sintoma = ['sintoma_1', 'sintoma_2', 'sintoma_3']

    for _, linha in mapa.iterrows():
        doenca = linha['doenca_associada']
        if doenca not in pontuacao:
            pontuacao[doenca] = {
                'score': 0,
                'nivel_risco': linha['nivel_risco'],
                'perfil': linha['perfil_predominante']
            }
        for col in colunas_sintoma:
            sintoma = normalizar(str(linha[col]))
            if sintoma in frase_norm:
                pontuacao[doenca]['score'] += 1

    # Filtra apenas doenças com ao menos uma correspondência
    candidatos = {k: v for k, v in pontuacao.items() if v['score'] > 0}

    if not candidatos:
        return 'Não identificado', 'indefinido', 'indefinido', []

    melhor = max(candidatos, key=lambda k: candidatos[k]['score'])
    sintomas = extrair_sintomas(frase, mapa)

    return (
        melhor,
        candidatos[melhor]['nivel_risco'],
        candidatos[melhor]['perfil'],
        sintomas
    )


print('Motor de diagnóstico carregado.')

## 5. Processamento dos 10 relatos

In [ ]:
resultados = []

for i, frase in enumerate(frases, 1):
    diagnostico, risco, perfil, sintomas = sugerir_diagnostico(frase, mapa)
    resultados.append({
        'paciente': i,
        'relato': frase,
        'sintomas_identificados': ', '.join(sintomas) if sintomas else 'nenhum',
        'diagnostico_sugerido': diagnostico,
        'nivel_risco': risco,
        'perfil_predominante': perfil
    })

df_resultados = pd.DataFrame(resultados)

# Exibição formatada
for _, row in df_resultados.iterrows():
    print(f"{'='*70}")
    print(f"Paciente {row['paciente']:02d}")
    print(f"Relato    : {row['relato']}")
    print(f"Sintomas  : {row['sintomas_identificados']}")
    print(f"Diagnóstico sugerido : {row['diagnostico_sugerido']}")
    print(f"Nível de risco       : {row['nivel_risco']}")
    print(f"Perfil predominante  : {row['perfil_predominante']}")
print('='*70)

## 6. Análise de viés de gênero na extração de sintomas

Esta seção verifica se o sistema identifica sintomas com a mesma eficácia  
para relatos com perfil masculino típico e feminino atípico.  

Essa análise conecta diretamente com o eixo de governança documentado na Fase 1:  
sistemas treinados com dados majoritariamente masculinos tendem a identificar  
sintomas atípicos femininos com menor precisão.

In [ ]:
# Frases 1-5: perfil masculino típico | Frases 6-10: perfil feminino atípico
df_resultados['grupo'] = df_resultados['paciente'].apply(
    lambda x: 'Masculino típico' if x <= 5 else 'Feminino atípico'
)

df_resultados['sintomas_count'] = df_resultados['sintomas_identificados'].apply(
    lambda x: len(x.split(',')) if x != 'nenhum' else 0
)

df_resultados['diagnostico_encontrado'] = df_resultados['diagnostico_sugerido'].apply(
    lambda x: 0 if x == 'Não identificado' else 1
)

analise = df_resultados.groupby('grupo').agg(
    media_sintomas_identificados=('sintomas_count', 'mean'),
    diagnosticos_encontrados=('diagnostico_encontrado', 'sum'),
    total_pacientes=('paciente', 'count')
).reset_index()

analise['taxa_diagnostico'] = (
    analise['diagnosticos_encontrados'] / analise['total_pacientes'] * 100
).round(1).astype(str) + '%'

print('Análise de viés de gênero na extração de sintomas:\n')
print(analise.to_string(index=False))

print('\nObservação:')
print('Se a taxa de diagnóstico for menor para o perfil feminino atípico,')
print('isso confirma o viés documentado na Fase 1: sintomas atípicos são')
print('menos representados em mapas de conhecimento clínico tradicionais.')

## 7. Exportação dos resultados

In [ ]:
df_resultados.to_csv('../data/resultados_diagnostico.csv', index=False, encoding='utf-8')
print('Resultados exportados para data/resultados_diagnostico.csv')

---

## Considerações finais

Este módulo demonstra como um sistema básico de NLP baseado em correspondência  
de padrões pode apoiar a triagem clínica inicial. As limitações são intencionais  
e fazem parte da análise:

- O sistema depende de correspondência exata de termos — não compreende semântica
- Sintomas atípicos (perfil feminino) são menos representados em mapas de conhecimento tradicionais
- Nas fases seguintes, modelos de ML e NLP mais sofisticados substituirão essa lógica de regras

**Este notebook não produz diagnósticos médicos. Tem finalidade exclusivamente acadêmica.**